# 2b. Provider catalog, policies and connectors (bearer token)

The Provider publishes what it offers as a DCAT catalog: datasets, their distributions (how the
data is delivered) and ODRL policies (the terms under which it may be used). A connector instance
bound to each distribution tells the Provider how to reach the real backend when a transfer
starts; in this notebook the connectors use a bearer token kept in the Keystore (`/api/v1/keystore/secrets`).

Steps:
1. Onboarding (`setup_auth.ipynb`), guarded.
2. Resolve the Provider's main catalog and data service (`/api/v1/catalog-agent/catalogs/main`, `/data-services/main`).
3. Create a dataset with a PULL distribution (`http+pull`) and a PUSH distribution (`http+push`) (`/api/v1/catalog-agent/datasets`, `/distributions`).
4. Attach an ODRL offer instantiated from a policy template (`/api/v1/catalog-agent/policy-templates`, `/instantiate-odrl-offer`).
5. Attach two ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`).
6. Store the backend's API key in the Provider's Keystore (`/api/v1/keystore/secrets`).
7. Register a PULL connector template (`BEARER_TOKEN`), instantiate it on the PULL distribution and inspect it (`/api/v1/connector/*`).
8. Register a PUSH connector template (`BEARER_TOKEN`), instantiate it on the PUSH distribution and inspect it.
9. The Consumer discovers the catalog and dataset over DSP (`/dsp/current/catalog/rpc/setup-catalog-request`, `setup-dataset-request`).

**Runs on its own.** It leaves `target_id`, `policy_id` and `data_space_provider_dsp_endpoint`
ready for the negotiation notebooks, and sets `catalog_setup = "bearer"` so that the transfer
notebooks know which variant is loaded. Each run creates a new dataset. `setup_catalog_policies.ipynb` is the
same flow with connectors without authentication.

## 1. Prerequisites

The guard runs the onboarding notebook only when this kernel has not run it yet. `%run` on a
notebook executes its cells in this same namespace, so the URLs, DIDs and `show` it defines are
available below.

In [1]:
if not globals().get("onboarded"):
    %run ./setup_auth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

## 2. Main catalog and data service (`/api/v1/catalog-agent/*`)

Every resource of the catalog is JSON-LD in the agent's own context
(`https://w3id.org/eunomia/v1/catalog-context.jsonld`): `@id` is its URN, texts are language maps
and version, visibility and owner travel in headers (`ETag`, `Eunomia-Visibility`). Every Provider
starts with a `main` catalog and a `main` data service. The data service is the
Provider's DSP endpoint; distributions point to it through `accessService`.

In [ ]:
catalog = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/catalogs/main", headers=json_header))
catalog_id = catalog["@id"]

In [ ]:
data_service = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/data-services/main", headers=json_header))
data_service_id = data_service["@id"]

## 3. Dataset and distributions (`/api/v1/catalog-agent/datasets`, `/distributions`)

A dataset is the unit of negotiation: a contract agreement always targets a dataset. Its
distributions are the ways to obtain it. The format of a distribution is the `format` a transfer
request asks for later:
- `http+pull`: the Consumer pulls the data from an endpoint the Provider hands over.
- `http+push`: the Provider pushes the data to a sink the Consumer names.

In [ ]:
# A Dataset in the catalog's JSON-LD context: texts are language maps, `inCatalog` places it.
payload = {
    "@context": "https://w3id.org/eunomia/v1/catalog-context.jsonld",
    "@type": "Dataset",
    "title": {"en": "My dataspace dataset", "es": "Mi dataset del espacio de datos"},
    "description": {"en": "Readings shared through the dataspace"},
    "keyword": {"en": ["tutorial", "readings"]},
    "theme": ["http://publications.europa.eu/resource/authority/data-theme/TECH"],
    "inCatalog": catalog_id,
}
dataset = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/datasets", headers=json_header, json=payload))
dataset_id = dataset["@id"]

In [ ]:
# `format` is the DSP transfer format (`dct:format`); `ofDataset` places it under the dataset.
payload = {
    "@type": "Distribution",
    "title": {"en": "Distribution of my dataset"},
    "description": {"en": "Pull access to my dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+pull",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_pull_id = distribution["@id"]

In [ ]:
payload = {
    "@type": "Distribution",
    "title": {"en": "Distribution of a streaming dataset"},
    "description": {"en": "Push delivery of my streaming dataset"},
    "accessService": [data_service_id],
    "ofDataset": dataset_id,
    "format": "http+push",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_push_id = distribution["@id"]

## 4. ODRL offer from a policy template (`/api/v1/catalog-agent/policy-templates`)

Policy templates are parameterised ODRL policies. The agent seeds `policy-1`, whose single
permission constrains `bla` to the value of the `$bla` parameter. Instantiating it attaches the
resulting offer to the dataset.

In [7]:
templates = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/policy-templates", headers=json_header))

HTTP 200 GET http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates
{
  "items": [
    {
      "author": "",
      "content": {
        "permission": [
          {
            "action": "use",
            "constraint": [
              {
                "leftOperand": "bla",
                "operator": "eq",
                "rightOperand": "$bla"
              }
            ]
          }
        ]
      },
      "date": "2026-10-16T21:16:00.293532Z",
      "description": "Mi example super policy",
      "id": "policy-1",
      "parameters": {
        "$bla": {
          "dataType": "string",
          "restrictions": {},
          "ui": {
            "hidden": false
          }
        }
      },
      "title": "Mi example policy",
      "userId": "system",
      "userRole": "/admin",
      "version": "1.0",
      "visibility": "Public"
    }
  ],
  "total": 1
}


In [8]:
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {
        "$bla": "did:web:university.upm.es"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants use to did:web:university.upm.es.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, json=payload))

HTTP 202 POST http://127.0.0.1:1200/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer
{
  "createdAt": "2026-10-17T14:00:14.832687Z",
  "description": "Research access for UPM: grants use to did:web:university.upm.es.",
  "entity": "urn:dataset:cd5ec7c5-5406-4510-8742-04e780decf8f",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:051c0acb-719c-4af9-a306-59d059afc701",
  "instantiationParameters": {
    "$bla": "did:web:university.upm.es"
  },
  "odrlOffer": {
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "bla",
            "operator": "eq",
            "rightOperand": "did:web:university.upm.es"
          }
        ]
      }
    ]
  },
  "sourceTemplateId": "policy-1",
  "sourceTemplateVersion": "1.0",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private"
}


## 5. ODRL offers written by hand (`/api/v1/catalog-agent/odrl-policies`)

An offer can also be posted as plain ODRL: permissions, obligations (duties) and prohibitions,
each with optional constraints. The Consumer will see all three offers on the dataset.

In [9]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "use",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-01-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-12-31T23:59:59Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"},
            ],
        }],
        "obligation": [{
            "action": "attribute",
            "constraint": [{"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}],
        }],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "createdAt": "2026-10-17T14:00:14.852829Z",
  "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
  "entity": "urn:dataset:cd5ec7c5-5406-4510-8742-04e780decf8f",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:c2ba3ee3-ec1b-4778-9410-45e62f362809",
  "instantiationParameters": null,
  "odrlOffer": {
    "obligation": [
      {
        "action": "attribute",
        "constraint": [
          {
            "leftOperand": "recipient",
            "operator": "isA",
            "rightOperand": "public"
          }
        ]
      }
    ],
    "permission": [
      {
        "action": "use",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-01-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
         

In [10]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "read",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-03-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-09-01T00:00:00Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "research"},
                {"leftOperand": "count", "operator": "lteq", "rightOperand": "10000"},
            ],
        }],
        "obligation": [],
        "prohibition": [
            {"action": "distribute", "constraint": []},
            {"action": "reproduce", "constraint": []},
        ],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/catalog-agent/odrl-policies
{
  "createdAt": "2026-10-17T14:00:14.878642Z",
  "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
  "entity": "urn:dataset:cd5ec7c5-5406-4510-8742-04e780decf8f",
  "entityType": "Dataset",
  "id": "urn:odrl-policy:ec383ce0-4ac3-475c-a5d8-89d14cd8213f",
  "instantiationParameters": null,
  "odrlOffer": {
    "obligation": [],
    "permission": [
      {
        "action": "read",
        "constraint": [
          {
            "leftOperand": "dateTime",
            "operator": "gteq",
            "rightOperand": "2026-03-01T00:00:00Z"
          },
          {
            "leftOperand": "dateTime",
            "operator": "lteq",
            "rightOperand": "2026-09-01T00:00:00Z"
          },
          {
            "leftOperand": "purpose",
            "operator": "isA",
            "rightOperand": "research"
          },
        

## 6. Backend secret in the Keystore (`/api/v1/keystore/secrets`)

The backend behind the connectors expects a bearer token. Instead of writing it into the
connector, it is stored once in the Provider's Keystore under `/ecostars/api_key`; templates refer
to it as `{{__RUNTIME_SECRET_{/ecostars/api_key}__}}` and the agent resolves it on each call.
Set `BACKEND_API_KEY` to store your own token.

A key is created once (`POST /api/v1/keystore/secrets`): on a rerun the cell replaces its value
instead (`PUT /api/v1/keystore/secrets/{key}`), naming the version it replaces (`expectedVersion`),
as the Keystore requires.

In [11]:
secret_key = "/ecostars/api_key"
secret_url = f"{data_space_provider}/api/v1/keystore/secrets"
secret_value = os.environ.get("BACKEND_API_KEY", "demo-backend-api-key")

existing = requests.get(f"{secret_url}{secret_key}", headers=json_header)
if existing.status_code == 404:
    payload = {"key": secret_key, "value": secret_value, "description": "Ecostars API key"}
    secret = show(requests.post(secret_url, headers=json_header, json=payload))
else:
    payload = {"value": secret_value, "expectedVersion": show(existing)["version"], "description": "Ecostars API key"}
    secret = show(requests.put(f"{secret_url}{secret_key}", headers=json_header, json=payload))

HTTP 200 GET http://127.0.0.1:1200/api/v1/keystore/secrets/ecostars/api_key
{
  "key": "/ecostars/api_key",
  "version": 5,
  "value": "*****",
  "description": "Ecostars API key",
  "createdAt": "2026-10-17T13:22:12.052697Z",
  "updatedAt": "2026-10-17T14:00:04.922481Z",
  "createdBy": "",
  "updatedBy": ""
}
HTTP 200 PUT http://127.0.0.1:1200/api/v1/keystore/secrets/ecostars/api_key
{
  "version": 6
}


## 7. PULL connector (`/api/v1/connector/*`)

A connector template describes how to reach a backend, with `{{__NAME__}}` placeholders; an
instance fills the parameters and binds the template to a distribution. In PULL mode the template
describes the data access the Consumer gets once the transfer starts.

In [12]:
payload = {
    "authentication": {
        "type": "BEARER_TOKEN",
        "token": {
            "type": "PLAIN",
            "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
        }
    },
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "Url", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "name": "urn:connector-template:80e733c9-47e4-4798-9d92-64e4c285c7ad",
  "author": "admin",
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:14.933321Z",
  "authentication": {
    "type": "BEARER_TOKEN",
    "token": {
      "type": "PLAIN",
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
    }
  },
  "interaction": {
    "mode": "PULL",
    "dataAccess": {
      "protocol": "HTTP",
      "urlTemplate": "{{__ACCESS_URL__}}",
      "method": "{{__ACCESS_METHODS__}}",
      "headers": "{{__ACCESS_HEADERS__}}",
      "bodyTemplate": null
    }
  },
  "parameters": [
    {
      "name": "ACCESS_URL",
      "title": "Url",
      "description": null,
      "paramType": "STRING",
      "required": true,
      "defaultValue": null
    },
    {
      "name": "ACCESS_METHODS",
      "title": "Methods",
      "description": null,
      "paramType": "VEC<STRING>",
      "required": true,
   

In [13]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "Pull access to the demo backend",
        "ownerId": "tutorial"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {"Content-type": "application/json"}
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
pull_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_pull_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "id": "urn:connector-instance:f28d45b9-f241-42fa-b209-383e10f9a8d1",
  "userId": "system",
  "name": "urn:connector-template:80e733c9-47e4-4798-9d92-64e4c285c7ad",
  "author": null,
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:14.961902Z",
  "authenticationConfig": {
    "type": "BEARER_TOKEN",
    "token": {
      "type": "PLAIN",
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
    }
  },
  "interaction": {
    "mode": "PULL",
    "dataAccess": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8081",
      "method": [
        "GET",
        "HEAD"
      ],
      "headers": {
        "Content-type": "application/json"
      },
      "bodyTemplate": null
    }
  },
  "distributionId": "urn:distribution:c4f3ec53-52c5-4e05-a4eb-e8d12a53f408"
}
HTTP 200 GET http://127.0.0.1:1200/api/v1/connector/instances/distribution/urn:distribution:c4f3ec53-52c5-4e05-a4eb-e8d1

## 8. PUSH connector (`/api/v1/connector/*`)

In PUSH mode the template says how to subscribe the Provider to the backend when a transfer
starts, and how to unsubscribe when it stops. `SUB_URL` and `UNSUB_URL` are filled by the
instance; the `SYS_*` and `RUNTIME_*` placeholders are filled by the agent at run time (its own
URL, the ingress of the transfer, and the subscription id the backend answered).

In [14]:
payload = {
    "authentication": {
        "type": "BEARER_TOKEN",
        "token": {
            "type": "PLAIN",
            "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
        }
    },
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription URL", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/templates
{
  "name": "urn:connector-template:762617f8-9aaa-4962-bb86-c6cb8aa2f3cd",
  "author": "admin",
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:14.989151Z",
  "authentication": {
    "type": "BEARER_TOKEN",
    "token": {
      "type": "PLAIN",
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
    }
  },
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "protocol": "HTTP",
      "urlTemplate": "{{__SUB_URL__}}",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{\"url\": \"{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
    },
    "unsubscribe": {
      "protocol": "HTTP",
      "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{}"
    }
  },
  "parameters": [
    {
      "name": "SUB_URL",
      "title": "S

In [15]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL": "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))
push_connector = show(requests.get(f"{data_space_provider}/api/v1/connector/instances/distribution/{distribution_push_id}", headers=json_header))

HTTP 200 POST http://127.0.0.1:1200/api/v1/connector/instances
{
  "id": "urn:connector-instance:38130d7b-c0c6-41a4-a56b-80517acf4863",
  "userId": "system",
  "name": "urn:connector-template:762617f8-9aaa-4962-bb86-c6cb8aa2f3cd",
  "author": null,
  "description": null,
  "version": "1.0",
  "createdAt": "2026-10-17T14:00:15.016310Z",
  "authenticationConfig": {
    "type": "BEARER_TOKEN",
    "token": {
      "type": "PLAIN",
      "content": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}"
    }
  },
  "interaction": {
    "mode": "PUSH",
    "subscribe": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8082/subscriptions/subscribe",
      "method": "POST",
      "headers": {},
      "bodyTemplate": "{\"url\": \"http://host.docker.internal:1200{{__RUNTIME_INGRESS__}}\", \"event_type\": \"hotel_waste_generated\"}"
    },
    "unsubscribe": {
      "protocol": "HTTP",
      "urlTemplate": "http://127.0.0.1:8082/subscriptions/unsubscribe/{{__RUNTIME_JSON_{.subscription.da

## 9. Catalog and dataset discovery by the Consumer (DSP 6.2.1 & DSP 6.2.2)

The Consumer asks its own agent to fetch the Provider's catalog
(`POST /dsp/current/catalog/rpc/setup-catalog-request`, sending `CatalogRequestMessage`) and
individual dataset (`POST /dsp/current/catalog/rpc/setup-dataset-request`, sending
`DatasetRequestMessage`). From the answer we keep:
- `target_id`: the `@id` of the dataset created above.
- `policy_id`: the `@id` of one of its offers. A negotiation must open on an offer the catalog
  publishes, on the dataset that publishes it.
- `data_space_provider_dsp_endpoint`: the `endpointURL` of the Provider's data service (`service`
  is a list).

In [ ]:
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcCatalogRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "noCache": True,
}
provider_catalog = show(requests.post(f"{data_space_consumer}/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, json=payload))["response"]

In [ ]:
dataset_rpc = show(requests.post(
    f"{data_space_consumer}/dsp/current/catalog/rpc/setup-dataset-request",
    headers=json_header,
    json={
        "@context": "https://w3id.org/eunomia/v1/context.jsonld",
        "@type": "RpcDatasetRequestMessage",
        "associatedAgentPeer": data_space_provider_participant_id,
        "dataset": dataset_id,
    },
))["response"]

published = next(d for d in provider_catalog["dataset"] if d["@id"] == dataset_id)
assert dataset_rpc["@id"] == published["@id"]

target_id = published["@id"]
policy_id = published["hasPolicy"][0]["@id"]
data_space_provider_dsp_endpoint = provider_catalog["service"][0]["endpointURL"]

# Marker read by the guards of the negotiation and transfer notebooks
catalog_setup = "bearer"

print(f"Dataset (target):      {target_id}")
print(f"Offer (policy):        {policy_id}")
print(f"Provider DSP address:  {data_space_provider_dsp_endpoint}")